LangChain is an open source framework with a pre-built agent architecture and integrations for any model or tool — so you can build agents that adapt as fast as the ecosystem evolves


In [ ]:
pip install -U langchain


In [ ]:
pip install langchain-groq


In [4]:
 GROQ_API_KEY="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"


In [5]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0,
    max_retries=2,
    # other params...
)

c:\Users\chint\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
messages = [
    ("system", "You are a helpful translator. Translate the user sentence to French."),
    ("human", "I love programming."),
]
model.invoke(messages[0])

AIMessage(content='The user sentence is: "system"\n\nThe translation of "system" to French is: "système"', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 54, 'total_tokens': 78, 'completion_time': 0.032235206, 'completion_tokens_details': None, 'prompt_time': 0.003715497, 'prompt_tokens_details': None, 'queue_time': 0.046508383, 'total_time': 0.035950703}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cbe62-9377-7423-b420-edc18e421d0d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 54, 'output_tokens': 24, 'total_tokens': 78})

In [12]:
pip install newsapi-python langgraph

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.agents.middleware import wrap_tool_call

from newsapi import NewsApiClient

newsapi = NewsApiClient(api_key='c4a43ae1a58740e4940e4bb06c7390f9')




@wrap_tool_call
def handle_tool_errors(request, handler):
    """Handle tool execution errors with custom messages."""
    try:
        return handler(request)
    except Exception as e:
        # Return a custom error message to the model
        return ToolMessage(
            content=f"Tool error: Please check your input and try again. ({str(e)})",
            tool_call_id=request.tool_call["id"]
        )


@tool
def get_news(query: str) -> str:
    """Get news information for a query."""
    print("get_news called with query:", query)
    response = newsapi.get_top_headlines(q=query, language='en')
    articles = response['articles']
    if not articles:
        return f"No news found for {query}."
    headlines = [article['title'] for article in articles[:5]]
    return f"News for {query}: {', '.join(headlines)}."


agent = create_agent(model, tools=[get_news] , middleware=[handle_tool_errors])

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's happening in iran war?"}]}
)

print(result["messages"][-1].content)

get_news called with query: iran war latest news


BadRequestError: Error code: 400 - {'error': {'message': "tool call validation failed: attempted to call tool 'brave_search' which was not in request.tools", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=brave_search>{"query": "iran war latest news"}</function>'}}